# Document Classification Pipeline - Orchestration Verification

This notebook demonstrates the simplified orchestration of the EKB pipeline using the new `run()` method.

In [5]:
import sys
import os
from google.cloud import storage

sys.path.append("../../..")

from pipelines.enterprise_knowledge_base import ClassificationPipeline

print("Unified Pipeline service imported.")

Unified Pipeline service imported.


## 1. Ingestion Helper
Upload a file to the landing zone with metadata.

In [6]:
def ingest_local_file(local_path: str, metadata: dict) -> str:
    client = storage.Client()
    bucket_name = "ag-core-dev-fdx7-kb-landing-zone"
    filename = os.path.basename(local_path)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)
    blob.metadata = metadata
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{filename}"

In [ ]:
pipeline = ClassificationPipeline()

# 1. Ingest
local_path = "../../../SoW - Innovation.pdf"
metadata = {
    "project": "orchestration-test",
    "domain": "hr",
    "trust-level": "archived",
    "uploader": "tester@example.com"
}

if os.path.exists(local_path):
    landing_uri = ingest_local_file(local_path, metadata)
    print(f"Ingested: {landing_uri}")

    # 2. Run Orchestration
    print("Starting Pipeline Orchestration...")
    result = pipeline.run(landing_uri)

    print("\n--- Pipeline Result ---")
    print(f"Final Domain: {result.final_domain}")
    print(f"Security Tier: {result.final_security_tier}")
    print(f"Sanitized URI: {result.final_sanitized_uri}")
else:
    print("Sample file not found.")

2026-04-24 22:44:36.611 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:run:54 - Starting pipeline orchestration for: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
2026-04-24 22:44:36.611 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.pipeline:_get_blob_metadata:190 - Extracting metadata for: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
2026-04-24 22:44:36.612 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gcs_service.service:get_blob_metadata:31 - Extracting detailed GCS metadata for: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
2026-04-24 22:44:36.612 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.gcs_service.service:_parse_uri:132 - Parsing GCS URI into dictionary: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf


Ingested: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
Starting Pipeline Orchestration...


2026-04-24 22:44:36.970 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:dlp_trigger:158 - Triggering DLP scan for: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
2026-04-24 22:44:36.971 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:38 - Starting DLP scan for: gs://ag-core-dev-fdx7-kb-landing-zone/Carta_recomendación_CONUEE.pdf
2026-04-24 22:44:36.971 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:_get_custom_keywords_config:209 - Building custom keywords config for inspection.
2026-04-24 22:44:37.895 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:66 - DLP Job created: projects/ag-core-dev-fdx7/locations/global/dlpJobs/i-4687486939820895586
2026-04-24 22:44:38.008 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:wait_f


--- Pipeline Result ---
Final Domain: hr
Security Tier: 1
Sanitized URI: gs://kb-hr/public/orchestration-test/tester/Carta_recomendación_CONUEE_masked.pdf


In [9]:
result.model_dump()

{'final_original_uri': 'gs://kb-hr/public/orchestration-test/tester/Carta_recomendación_CONUEE.pdf',
 'final_sanitized_uri': 'gs://kb-hr/public/orchestration-test/tester/Carta_recomendación_CONUEE_masked.pdf',
 'final_security_tier': 1,
 'final_domain': 'hr'}